## Import libraries

In [1]:
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import least_squares
import plotly.graph_objects as go
import plotly.express as px

## Load data - exports

In [2]:
ROOT = Path.cwd().parent

data_path = ROOT / 'data'

In [3]:
df = pd.read_csv(data_path / 'dati_assoluti.csv')
df = df.reset_index(drop=True)

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Time'], y=df['China'], mode='lines', name='China'))
fig.add_trace(go.Scatter(x=df['Time'], y=df['Europe'], mode='lines', name='Europe'))
fig.add_trace(go.Scatter(x=df['Time'], y=df['JP+KR'], mode='lines', name='JP+KR'))
fig.add_trace(go.Scatter(x=df['Time'], y=df['SoutheastAsia'], mode='lines', name='SoutheastAsia'))
fig.add_trace(go.Scatter(x=df['Time'], y=df['Taiwan'], mode='lines', name='Taiwan'))
fig.update_layout(
    title='Exports to USA by Country',
    xaxis_title='Time',
    yaxis_title='Exports',
    width=1000,
    height=500,
    template='simple_white'
)
fig.show()

## Model

In [5]:
# Import normalized data
df_normalized = pd.read_csv(data_path / 'dati_normalizzati.csv')
df_normalized = df_normalized.reset_index(drop=True)
df_normalized = df_normalized.drop(columns=['Europe', 'total'])
df_normalized

,Time,China,JP+KR,SoutheastAsia,Taiwan
0,2002-01-01,0.110311,0.380993,0.352239,0.054684
1,2002-02-01,0.093534,0.454315,0.287546,0.057884
2,2002-03-01,0.107718,0.406185,0.349512,0.052608
3,2002-04-01,0.123506,0.467669,0.310324,0.044436
4,2002-05-01,0.110933,0.290828,0.515878,0.033958
...,...,...,...,...,...
235,2021-08-01,0.062298,0.204443,0.632360,0.041596
236,2021-09-01,0.055998,0.177974,0.663774,0.046366
237,2021-10-01,0.038418,0.136539,0.749306,0.035138
238,2021-11-01,0.054520,0.176605,0.676466,0.040333


In [6]:
# Initialize the DataFrame for tariffs
df_tariffs = pd.DataFrame()
df_tariffs['Time'] = df_normalized['Time']
df_tariffs

,Time
0,2002-01-01
1,2002-02-01
2,2002-03-01
3,2002-04-01
4,2002-05-01
...,...
235,2021-08-01
236,2021-09-01
237,2021-10-01
238,2021-11-01


In [7]:
# Create tariffs columns based on the specified conditions
df_tariffs['China Tariff'] = np.where(df_tariffs['Time'] < '2012-05-01', 0, 1)
df_tariffs['JP+KR Tariff'] = np.where(df_tariffs['Time'] < '2018-02-01', 0, 1)
df_tariffs['SoutheastAsia Tariff'] = np.where(df_tariffs['Time'] < '2018-02-01', 0, 1)
df_tariffs['Taiwan Tariff'] = np.where(df_tariffs['Time'] < '2014-12-01', 0, 1)
df_tariffs

,Time,China Tariff,JP+KR Tariff,SoutheastAsia Tariff,Taiwan Tariff
0,2002-01-01,0,0,0,0
1,2002-02-01,0,0,0,0
2,2002-03-01,0,0,0,0
3,2002-04-01,0,0,0,0
4,2002-05-01,0,0,0,0
...,...,...,...,...,...
235,2021-08-01,1,1,1,1
236,2021-09-01,1,1,1,1
237,2021-10-01,1,1,1,1
238,2021-11-01,1,1,1,1


In [8]:
# Define the Lotka-Volterra model with tariffs
def lotka_volterra_with_tariffs_growth(t, X, tariffs,
                                r1, r2, r3, r4, # Growth rates
                                g1, g2, g3, g4, # Growth reduction factors
                                a11, a12, a13, a14, # Competition coefficients for X1
                                a21, a22, a23, a24, # Competition coefficients for X2
                                a31, a32, a33, a34, # Competition coefficients for X3
                                a41, a42, a43, a44, # Competition coefficients for X4
                                # Tariff impact coefficients
                                b11, b12, b13, b14,
                                b21, b22, b23, b24,
                                b31, b32, b33, b34,
                                b41, b42, b43, b44):
    X1, X2, X3, X4 = X # Unpack the state variables
    tariff_1, tariff_2, tariff_3, tariff_4 = tariffs.iloc[int(t), 1:] # Extract tariffs for the current time step

    dx1dt = r1 * (1 - g1 * tariff_1) * X1 * (1 - a11*X1 - a12*X2 - a13*X3 - a14*X4) - b11 * X1 * tariff_1 + b12 * X2 * tariff_2 + b13 * X3 * tariff_3 + b14 * X4 * tariff_4
    dx2dt = r2 * (1 - g2 * tariff_2) * X2 * (1 - a21*X1 - a22*X2 - a23*X3 - a24*X4) + b21 * X1 * tariff_1 - b22 * X2 * tariff_2 + b23 * X3 * tariff_3 + b24 * X4 * tariff_4
    dx3dt = r3 * (1 - g3 * tariff_3) * X3 * (1 - a31*X1 - a32*X2 - a33*X3 - a34*X4) + b31 * X1 * tariff_1 + b32 * X2 * tariff_2 - b33 * X3 * tariff_3 + b34 * X4 * tariff_4
    dx4dt = r4 * (1 - g4 * tariff_4) * X4 * (1 - a41*X1 - a42*X2 - a43*X3 - a44*X4) + b41 * X1 * tariff_1 + b42 * X2 * tariff_2 + b43 * X3 * tariff_3 - b44 * X4 * tariff_4

    return [dx1dt, dx2dt, dx3dt, dx4dt]

In [9]:
# Define the parameters for the model
# These parameters can be adjusted based on the model's requirements
params = {
    'r1': 0.5, 'r2': 0.3, 'r3': 0.4, 'r4': 0.35,
    'g1': 0.1, 'g2': 0.1, 'g3': 0.1, 'g4': 0.1,
    'a11': 1.0, 'a12': 0.6, 'a13': 0.2, 'a14': 0.3,
    'a21': 0.3, 'a22': 1.0, 'a23': 0.5, 'a24': 0.4,
    'a31': 0.4, 'a32': 0.2, 'a33': 1.0, 'a34': 0.5,
    'a41': 0.2, 'a42': 0.3, 'a43': 0.4, 'a44': 1.0,
    'b11': 0.1, 'b12': 0.1, 'b13': 0.1, 'b14': 0.1,
    'b21': 0.1, 'b22': 0.1, 'b23': 0.1, 'b24': 0.1,
    'b31': 0.1, 'b32': 0.1, 'b33': 0.1, 'b34': 0.1,
    'b41': 0.1, 'b42': 0.1, 'b43': 0.1, 'b44': 0.1
}

# Initial conditions
X0 = df_normalized.iloc[0, 1:].values.tolist()  # Use the first row of normalized data as initial conditions

# Time span and evaluation points
t_span = (0, 239)
t_eval = np.linspace(*t_span)

# Solve the system
sol = solve_ivp(
    fun=lambda t, X: lotka_volterra_with_tariffs_growth(t, X, df_tariffs, **params),
    t_span=t_span,
    y0=X0,
    t_eval=t_eval
)

# Plot the results using plotly.graph_objects
fig = go.Figure()
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], mode='lines', name='China'))
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[1], mode='lines', name='JP+KR'))
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[2], mode='lines', name='SoutheastAsia'))
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[3], mode='lines', name='Taiwan'))

fig.update_layout(
    title='Lotka-Volterra Competition Model — Solar Exports to the US',
    xaxis_title='Time (months)',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    width=1000,
    height=500,
    template='simple_white'
)
fig.show()

## Model Calibration - Least Squares

In [10]:
# Extract data for each region
china_data = df_normalized['China']
jpkr_data = df_normalized['JP+KR']
southeastasia_data = df_normalized['SoutheastAsia']
taiwan_data = df_normalized['Taiwan']

# Stack data for comparison
data = np.vstack([china_data, jpkr_data, southeastasia_data, taiwan_data])  # shape (4, T)
t_data = np.linspace(0, len(china_data)-1, len(china_data))  # e.g. months

In [11]:
# Define the residuals function for optimization
def residuals(params, X0, tariffs, t_data, data):
    # Unpack the parameters
    r1, r2, r3, r4 = params[:4]
    g1, g2, g3, g4 = params[4:8]
    a11, a12, a13, a14 = params[8:12]
    a21, a22, a23, a24 = params[12:16]
    a31, a32, a33, a34 = params[16:20]
    a41, a42, a43, a44 = params[20:24]
    b11, b12, b13, b14 = params[24:28]
    b21, b22, b23, b24 = params[28:32]
    b31, b32, b33, b34 = params[32:36]
    b41, b42, b43, b44 = params[36:40]

    # Solve the system using the provided parameters
    sol = solve_ivp(
        fun=lambda t, X: lotka_volterra_with_tariffs_growth(
            t, X, tariffs, r1, r2, r3, r4,
            g1, g2, g3, g4,
            a11, a12, a13, a14,
            a21, a22, a23, a24,
            a31, a32, a33, a34,
            a41, a42, a43, a44,
            b11, b12, b13, b14,
            b21, b22, b23, b24,
            b31, b32, b33, b34,
            b41, b42, b43, b44
        ),
        t_span=(t_data[0], t_data[-1]),
        y0=X0,
        t_eval=t_data
    )

    # Check the length of the solution
    if sol.y.shape[1] != len(t_data):
        return np.ones(data.size) * 1e6

    return (sol.y - data).ravel()

In [12]:
# Initial guess for 4 regions (total 40 params)
initial_guess = [
    params['r1'], params['r2'], params['r3'], params['r4'],
    params['g1'], params['g2'], params['g3'], params['g4'],
    params['a11'], params['a12'], params['a13'], params['a14'],
    params['a21'], params['a22'], params['a23'], params['a24'],
    params['a31'], params['a32'], params['a33'], params['a34'],
    params['a41'], params['a42'], params['a43'], params['a44'],
    params['b11'], params['b12'], params['b13'], params['b14'],
    params['b21'], params['b22'], params['b23'], params['b24'],
    params['b31'], params['b32'], params['b33'], params['b34'],
    params['b41'], params['b42'], params['b43'], params['b44']
]

In [13]:
# Initial export levels
X0 = data[:, 0]

# Fit the model
result = least_squares(residuals, initial_guess, args=(X0, df_tariffs, t_data, data), bounds=(0, np.inf))  # constrain params to be positive

# Extract best-fit parameters
fitted_params = result.x

In [14]:
fitted_params

array([0.63770708, 0.39260002, 0.5232244 , 0.37974133, 0.25328867,
       0.06794388, 0.88138626, 0.16705785, 1.04832274, 0.92252959,
       1.41104384, 0.0100829 , 0.91854037, 0.57142079, 1.5506745 ,
       1.98547199, 0.71337915, 0.68351752, 1.55426804, 1.85829001,
       0.94543724, 0.36084363, 1.91186143, 1.23696495, 0.0265728 ,
       0.01245067, 0.00564682, 0.06715551, 0.02046286, 0.1148082 ,
       0.05805264, 0.03389762, 0.01032545, 0.08899053, 0.00653549,
       0.13462847, 0.01903264, 0.014109  , 0.00601231, 0.15121972])

In [15]:
# Solve the system with the fitted parameters
t_data = np.linspace(0, len(china_data)-1, len(china_data))
sol = solve_ivp(
    fun=lambda t, X: lotka_volterra_with_tariffs_growth(t, X, df_tariffs, *fitted_params),
    t_span=(t_data[0], t_data[-1]),
    y0=X0,
    t_eval=t_data
)

In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_data[:len(data[0])], y=data[0], mode='markers', name='China (data)', marker=dict(color='red')))
fig.add_trace(go.Scatter(x=t_data, y=sol.y[0], mode='lines', name='China (fit)', line=dict(color='red')))

fig.update_layout(
    title='Lotka-Volterra with Tariffs - Calibration',
    xaxis_title='Time',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    template='simple_white'
)
fig.show()

In [17]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_data[:len(data[1])], y=data[1], mode='markers', name='JP+KR (data)', marker=dict(color='blue')))
fig.add_trace(go.Scatter(x=t_data, y=sol.y[1], mode='lines', name='JP+KR (fit)', line=dict(color='blue')))

fig.update_layout(
    title='Lotka-Volterra with Tariffs - Calibration',
    xaxis_title='Time',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    template='simple_white'
)
fig.show()

In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_data[:len(data[2])], y=data[2], mode='markers', name='SoutheastAsia (data)', marker=dict(color='green')))
fig.add_trace(go.Scatter(x=t_data, y=sol.y[2], mode='lines', name='SoutheastAsia (fit)', line=dict(color='green')))

fig.update_layout(
    title='Lotka-Volterra with Tariffs - Calibration',
    xaxis_title='Time',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    template='simple_white'
)
fig.show()

In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_data[:len(data[3])], y=data[3], mode='markers', name='Taiwan (data)', marker=dict(color='purple')))
fig.add_trace(go.Scatter(x=t_data, y=sol.y[3], mode='lines', name='Taiwan (fit)', line=dict(color='purple')))

fig.update_layout(
    title='Lotka-Volterra with Tariffs - Calibration',
    xaxis_title='Time',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    template='simple_white'
)
fig.show()

## Prediction

In [20]:
# Create a new DataFrame for future periods, assuming an active tariff for Southeast Asia only
new_periods = pd.DataFrame(np.zeros([120, 5]), columns=['Time', 'China Tariff', 'JP+KR Tariff', 'SoutheastAsia Tariff', 'Taiwan Tariff'])
new_periods['Time'] = pd.date_range(start='2022-01-01', periods=120, freq='MS')
new_periods['SoutheastAsia Tariff'] = np.ones(len(new_periods))
new_periods

,Time,China Tariff,JP+KR Tariff,SoutheastAsia Tariff,Taiwan Tariff
0,2022-01-01,0.0,0.0,1.0,0.0
1,2022-02-01,0.0,0.0,1.0,0.0
2,2022-03-01,0.0,0.0,1.0,0.0
3,2022-04-01,0.0,0.0,1.0,0.0
4,2022-05-01,0.0,0.0,1.0,0.0
...,...,...,...,...,...
115,2031-08-01,0.0,0.0,1.0,0.0
116,2031-09-01,0.0,0.0,1.0,0.0
117,2031-10-01,0.0,0.0,1.0,0.0
118,2031-11-01,0.0,0.0,1.0,0.0


In [21]:
# Concatenate the new periods with the existing tariffs DataFrame
df_tariffs_pred = pd.concat([df_tariffs.copy(), new_periods], axis=0)
df_tariffs_pred = df_tariffs_pred.reset_index(drop=True)
df_tariffs_pred['Time'] = pd.date_range(start='2002-01-01', periods=len(df_tariffs_pred), freq='MS')
df_tariffs_pred

,Time,China Tariff,JP+KR Tariff,SoutheastAsia Tariff,Taiwan Tariff
0,2002-01-01,0.0,0.0,0.0,0.0
1,2002-02-01,0.0,0.0,0.0,0.0
2,2002-03-01,0.0,0.0,0.0,0.0
3,2002-04-01,0.0,0.0,0.0,0.0
4,2002-05-01,0.0,0.0,0.0,0.0
...,...,...,...,...,...
355,2031-08-01,0.0,0.0,1.0,0.0
356,2031-09-01,0.0,0.0,1.0,0.0
357,2031-10-01,0.0,0.0,1.0,0.0
358,2031-11-01,0.0,0.0,1.0,0.0


In [22]:
# Solve the system using the fitted parameters and the new tariffs DataFrame
sol = solve_ivp(
    fun=lambda t, X: lotka_volterra_with_tariffs_growth(t, X, df_tariffs_pred, *fitted_params),
    t_span=(0, len(df_tariffs_pred) - 1),
    y0=X0,
    t_eval=np.arange(len(df_tariffs_pred))
)

In [23]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'][:len(data[0])], y=data[0],
    mode='markers', name='China (data)', marker=dict(color='red')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'], y=sol.y[0],
    mode='lines', name='China (predicted)', line=dict(color='red')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'][:len(data[1])], y=data[1],
    mode='markers', name='JP+KR (data)', marker=dict(color='blue')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'], y=sol.y[1],
    mode='lines', name='JP+KR (predicted)', line=dict(color='blue')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'][:len(data[2])], y=data[2],
    mode='markers', name='SoutheastAsia (data)', marker=dict(color='green')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'], y=sol.y[2],
    mode='lines', name='SoutheastAsia (predicted)', line=dict(color='green')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'][:len(data[3])], y=data[3],
    mode='markers', name='Taiwan (data)', marker=dict(color='purple')
))
fig.add_trace(go.Scatter(
    x=df_tariffs_pred['Time'], y=sol.y[3],
    mode='lines', name='Taiwan (predicted)', line=dict(color='purple')
))
fig.update_layout(
    title='Lotka-Volterra Prediction',
    xaxis_title='Time',
    yaxis_title='Normalized Export Level',
    legend=dict(x=0.01, y=0.99),
    template='simple_white'
)
fig.show()